## Behavorial Analysis With BIDSReader
The **BIDSReader** package helps CML researchers efficiently load BIDS data without having to deal with the BIDS syntax. This introduction will teach you how to load behavioral BIDS data using **BIDSReader** for analysis.

### Loading Study-wide Metadata
Once you set the study root, you can inquire simple metadata queries such a getting a list of subjects, tasks, and max number of sessions in the study.

In [1]:
# imports
import pandas as pd; pd.set_option('display.max_columns', None)
import numpy as np
import os 
import sys
sys.path.insert(0, 'dependencies/bidsreader')
from bidsreader import CMLBIDSReader as BIDSReader

# set the bids_root
bids_root = "/data/LTP_BIDS/"
reader = BIDSReader()
print(f"Scalp Subjects: {reader.get_dataset_subjects()}")
print(f"Scalp Tasks: {reader.get_dataset_tasks()}")
print(f"Max # of Sessions: {reader.get_dataset_max_sessions()}")

Scalp Subjects: ['LTP063', 'LTP064', 'LTP065', 'LTP066', 'LTP067', 'LTP068', 'LTP069', 'LTP070', 'LTP073', 'LTP074', 'LTP093', 'LTP106', 'LTP115', 'LTP117', 'LTP122', 'LTP123', 'LTP133', 'LTP138', 'LTP187', 'LTP207', 'LTP229', 'LTP246', 'LTP260', 'LTP312', 'LTP327', 'LTP329', 'LTP339', 'LTP347', 'LTP354', 'LTP355', 'LTP606', 'LTP607', 'LTP609', 'LTP610', 'LTP612', 'LTP613', 'LTP614', 'LTP615', 'LTP617', 'LTP618', 'LTP619', 'LTP620', 'LTP621', 'LTP622', 'LTP624', 'LTP625', 'LTP626', 'LTP627']
Scalp Tasks: ['ltpfr', 'ltpfr2', 'valuecourier', 'vcbehonly', 'vffr']
Max # of Sessions: 23


### Loading Subject Metadata
You can use those inquiries to choose a subject and inquire about the tasks and sessions the subject has performed.

In [2]:
subject = "LTP606"

# use set_fields to set multiple fields at once
reader.set_fields(subject=subject)
tasks_list = reader.get_subject_tasks()
sessions_list = reader.get_subject_sessions()
print(f"{subject} Tasks: {tasks_list}")
print(f"{subject} Sessions: {sessions_list}")

LTP606 Tasks: ['valuecourier']
LTP606 Sessions: ['0', '1', '2', '3', '4', '5']


## Example of some of the tasks ran by the lab by category:

### Verbal free-recall tasks (no-stim)
* FR1
* catFR1
* RepFR1

### Paired-associates tasks
* PAL1
* PAL2 (open-loop stim)
* PAL3 (closed-loop stim)
* PAL5 (closed-loop stim)

### Spatial navigation tasks
* YC1
* TH1
* THR
* THR1
* YC2 (open-loop stim)
* TH3 (closed-loop stim)
* EFRCourierReadOnly
* EFRCourierOpenLoop

### Verbal free-recall w/ stim
(Basically, any FR task with a number above 1 somewhere)
* FR2 (open-loop)
* catFR2
* FR3 (closed-loop)
* catFR3
* FR5 (closed-loop)
* catFR5
* PS4_FR (closed-loop)
* PS4_catFR (closed-loop)
* PS5_catFR (closed-loop)
* FR6 (multi-target stim)
* catFR6 (multi-target stim)
* TICL_FR (encoding/math/retrieval stim)
* RepFR2

### No-task stimulation ("parameter search")
* PS1
* PS2/PS2.1
* PS3
* LocationSearch
* OPS


## Loading Behavioral Events
After identifying the tasks and sessions completed by your chosen subject, you can now load the events.

### BIDSReader Fields
* root (str | Path: path to the study root) (Defaults to /data/LTP_BIDS/)
* subject (str: subject id)
* task (str: experiment id)
* session (str: session id)
* eeg_type (str: type of eeg used) (eeg or ieeg)
* acquisition (str: type of ieeg acquisition) (bipolar or monopolar)
* space (str: iEEG coordinate system) (MNI152NLin6ASym, Talarich) 

To load events, you must specify the subject, task, and session. eeg_type and space are necessary for BIDS, but will be infered based on the existing data.

In [3]:
# set task and session which will let BIDSReader infer eeg_type and space 
task = tasks_list[0]
session = sessions_list[0]
reader.set_fields(task=task, session=session)

CMLBIDSReader(root=PosixPath('/data/LTP_BIDS'), subject='LTP606', session='0', task='valuecourier', device='eeg', space='CapTrak')

## Loading Behavioral Data
As mentioned, there are two locations where the behavioral data is held, in the "beh" folder and in the "eeg/ieeg" folder. BIDSReader makes it easy to load from each by setting "event_type" in the "load_events" function to "beh" or "events". 

In [4]:
evs_beh = reader.load_events(event_type="beh")
evs_beh[:5]

,mstime,trial_type,stim_file,actualvalue,compensation,eegfile,eogArtifact,experiment,intruded,intrusion,item,itemno,itemvalue,montage,msoffset,multiplier,numingroupchosen,phase,playerrotY,presX,presZ,primacybuf,protocol,recalled,rectime,recencybuf,serialpos,session,store,storeX,storeZ,storepointtype,subject,trial,valuerecall
0,0,store mappings,NaN,NaN,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,NaN,NaN,NaN,NaN,0,-1,0.846154,4.0,1,NaN,-4.949219,10.5,2.0,ltp,NaN,NaN,3.0,NaN,0,NaN,NaN,NaN,NaN,LTP606,0,NaN
1,4142,VIDEO_START,NaN,NaN,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,NaN,NaN,-1.0,NaN,0,-1,0.846154,4.0,1,NaN,-4.949219,10.5,2.0,ltp,NaN,NaN,3.0,NaN,0,NaN,NaN,NaN,NaN,LTP606,0,NaN
2,217121,VIDEO_STOP,NaN,NaN,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,NaN,NaN,-1.0,NaN,0,-1,0.846154,4.0,1,NaN,-4.949219,10.5,2.0,ltp,NaN,NaN,3.0,NaN,0,NaN,NaN,NaN,NaN,LTP606,0,NaN
3,304052,SESS_START,NaN,NaN,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,NaN,NaN,NaN,NaN,0,-1,0.846154,4.0,1,NaN,-4.949219,10.5,2.0,ltp,NaN,NaN,3.0,NaN,0,NaN,NaN,NaN,NaN,LTP606,0,NaN
4,304098,TL_START,NaN,NaN,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,NaN,NaN,NaN,NaN,0,-1,0.846154,4.0,1,NaN,-4.949219,10.5,2.0,ltp,NaN,NaN,3.0,NaN,0,NaN,NaN,NaN,NaN,LTP606,0,NaN


In [5]:
evs_eeg = reader.load_events(event_type="eeg")
evs_eeg[:5]

,onset,duration,trial_type,sample,stim_file,actualvalue,compensation,eegfile,eogArtifact,experiment,intruded,intrusion,item,itemno,itemvalue,montage,msoffset,multiplier,numingroupchosen,phase,playerrotY,presX,presZ,primacybuf,protocol,recalled,rectime,recencybuf,serialpos,session,store,storeX,storeZ,storepointtype,subject,trial,valuerecall
0,11.445312,NaN,store mappings,23440,NaN,NaN,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,NaN,NaN,NaN,NaN,0,-1,0.846154,4.0,1,NaN,-4.949219,10.5,2.0,ltp,NaN,NaN,3.0,NaN,0,NaN,NaN,NaN,NaN,LTP606,0,NaN
1,15.586914,NaN,VIDEO_START,31922,NaN,NaN,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,NaN,NaN,-1.0,NaN,0,-1,0.846154,4.0,1,NaN,-4.949219,10.5,2.0,ltp,NaN,NaN,3.0,NaN,0,NaN,NaN,NaN,NaN,LTP606,0,NaN
2,228.548828,NaN,VIDEO_STOP,468068,NaN,NaN,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,NaN,NaN,-1.0,NaN,0,-1,0.846154,4.0,1,NaN,-4.949219,10.5,2.0,ltp,NaN,NaN,3.0,NaN,0,NaN,NaN,NaN,NaN,LTP606,0,NaN
3,315.473145,NaN,SESS_START,646089,NaN,NaN,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,NaN,NaN,NaN,NaN,0,-1,0.846154,4.0,1,NaN,-4.949219,10.5,2.0,ltp,NaN,NaN,3.0,NaN,0,NaN,NaN,NaN,NaN,LTP606,0,NaN
4,315.519043,NaN,TL_START,646183,NaN,NaN,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,NaN,NaN,NaN,NaN,0,-1,0.846154,4.0,1,NaN,-4.949219,10.5,2.0,ltp,NaN,NaN,3.0,NaN,0,NaN,NaN,NaN,NaN,LTP606,0,NaN


## evs/Events Data Frame

Before digging into the data, a little information about the experiment.

In free recall experiments, subjects memorize a series of word lists during an experimental session. Each list consists of an encoding period, during which the words (or "items") in the list are presented to the subject one by one, followed by a retrieval period during which the subject recalls as many words as they can in any order. The experiment is called "free" recall because the subjects are "free" to recall the items in any order; this in contrast to a serial recall experiment in which subjects must recall the items in the order they were presented.

Some free recall experiments contain a "distractor" period between the end of the encoding period and the beginning of the retrieval period. The purpose of the distractor is to "clear out" subjects' minds before starting to recall items; without the distractor, subjects are far more likely to recall items from the end of the list in what is known as the recency effect. In the experiments described here, the distractor period consists of a series of arithmetic problems of the form 'A + B + C', to which subjects respond by typing the answer. The FR1 experiment also begins each list by presenting the subject with a countdown period (10, 9, 8, ...).

The events dataframe contains information about everything that happened during an experimental session. It indicates the time at which every word appeared on the screen, and when those words were later recalled. It also contains information about events that you might not care about, such as when the countdown timer starts and ends.
<center>
<img src="https://github.com/esolomon/PythonBootcamp2019/blob/master/figures/task_design-01.jpg?raw=true" width=650>
</center>
Let's take a look at all the columns in this dataframe...

In [6]:
cols = set(evs_beh.columns) | set(evs_eeg.columns)
print(sorted(cols))

['actualvalue', 'compensation', 'duration', 'eegfile', 'eogArtifact', 'experiment', 'intruded', 'intrusion', 'item', 'itemno', 'itemvalue', 'montage', 'msoffset', 'mstime', 'multiplier', 'numingroupchosen', 'onset', 'phase', 'playerrotY', 'presX', 'presZ', 'primacybuf', 'protocol', 'recalled', 'recencybuf', 'rectime', 'sample', 'serialpos', 'session', 'stim_file', 'store', 'storeX', 'storeZ', 'storepointtype', 'subject', 'trial', 'trial_type', 'valuerecall']


... and here is what some of the important ones mean. 
* 'sample' indicates where (in samples) in the EEG file this event occurred. MNE-BIDS needs this info, but usually you won't need to deal with it directly.
* 'onset' indicates where (in seconds) where in the EEG file this event occurred.
* 'duration' indicates the length (in seconds) of the event. Currently NaN.
* 'answer' is the participants response to a math distractor problem.
* 'experiment' is the behavioral task we're looking at. 
* 'intrusion' is an indicator of intrusion events during the recall period. -1 indicates an extra-list intrusion, otherwise, number of lists prior from which the word came.
* 'intruded' indicates if an intrustion took place.  
* 'item_name' is the word that was presented or recalled.
* **'list' is the list number (in the LTPFR2 scalp EEG dataset list number is instead contained in the 'trial' column).** 
* 'mstime' is a time indicator, in ms. Good for comparing between events, but the absolute value is meaningless. 
* 'recalled' is a indicator of whether an encoding word was later recalled successfully.
* 'rectime' is the time, in ms, when a word was recalled relative to the start of the recall period for that list.
* 'serialpos' is the serial position of a presented/recalled word
* 'session' is the session number the data came from
* 'subject' is the subject you're analyzing!
* 'trial_type' is the type of event, e.g. 'WORD' or 'REC_WORD' --> 

Please see https://pennmem.github.io/cmlreaders/html/events.html for more information!

Two types of events that we are often interesting in analyzing are encoding ("WORD") events and recall ("REC_WORD") events. "WORD" is just all words and "REC_WORD" just includes correctly recalled. Know that not all words the participants say are correct. 

In [7]:
# let's filter for encoding events
word_evs = evs_beh[evs_beh['trial_type'] == 'WORD']
word_evs.head()

,mstime,trial_type,stim_file,actualvalue,compensation,eegfile,eogArtifact,experiment,intruded,intrusion,item,itemno,itemvalue,montage,msoffset,multiplier,numingroupchosen,phase,playerrotY,presX,presZ,primacybuf,protocol,recalled,rectime,recencybuf,serialpos,session,store,storeX,storeZ,storepointtype,subject,trial,valuerecall
68,2287892,WORD,wordpools/valuecourier_wordpool.txt,14.066667,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,3,ValueCourier,0.0,NaN,JEANS,NaN,10.0,0,-1,0.846154,4.0,1,84.570000,56.968750,81.7500,2.0,ltp,1.0,NaN,3.0,1.0,0,clothing_store,56.718750,77.93750,Temporal,LTP606,0,15.0
70,2352924,WORD,wordpools/valuecourier_wordpool.txt,14.066667,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,3,ValueCourier,0.0,NaN,WATER_GUN,NaN,13.0,0,-1,0.846154,4.0,1,103.199997,-33.375000,-83.8125,2.0,ltp,1.0,NaN,3.0,2.0,0,toy_store,-29.187500,-93.87500,Temporal,LTP606,0,15.0
71,2375559,WORD,wordpools/valuecourier_wordpool.txt,14.066667,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,3,ValueCourier,0.0,NaN,LIGHT,NaN,22.0,0,-1,0.846154,4.0,1,206.360001,22.296875,-51.3750,2.0,ltp,0.0,NaN,3.0,3.0,0,bike_shop,18.671875,-51.71875,Temporal,LTP606,0,15.0
73,2437993,WORD,wordpools/valuecourier_wordpool.txt,14.066667,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,3,ValueCourier,0.0,NaN,PEARLS,NaN,22.0,0,-1,0.846154,4.0,1,2.900000,-42.531250,61.0000,2.0,ltp,0.0,NaN,3.0,4.0,0,jewelry_store,-37.500000,61.00000,Temporal,LTP606,0,15.0
74,2454991,WORD,wordpools/valuecourier_wordpool.txt,14.066667,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,3,ValueCourier,0.0,NaN,NAILGUN,NaN,16.0,0,-1,0.846154,4.0,1,77.989998,-6.160156,125.7500,2.0,ltp,1.0,NaN,3.0,5.0,0,hardware_store,-8.867188,131.87500,Temporal,LTP606,0,15.0


In [8]:
# as well as recall events
rec_evs = evs_beh[evs_beh['trial_type'] == 'REC_WORD']
rec_evs.head()

,mstime,trial_type,stim_file,actualvalue,compensation,eegfile,eogArtifact,experiment,intruded,intrusion,item,itemno,itemvalue,montage,msoffset,multiplier,numingroupchosen,phase,playerrotY,presX,presZ,primacybuf,protocol,recalled,rectime,recencybuf,serialpos,session,store,storeX,storeZ,storepointtype,subject,trial,valuerecall
91,2765168,REC_WORD,wordpools/valuecourier_wordpool.txt,14.066667,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,0.0,DOUGHNUT,71.0,NaN,0,20,0.846154,4.0,1,NaN,-4.765625,10.03125,2.0,ltp,NaN,1175.0,3.0,14.0,0,bakery,-80.687500,110.125000,Temporal,LTP606,0,15.0
92,2766506,REC_WORD,wordpools/valuecourier_wordpool.txt,14.066667,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,0.0,WATER_GUN,232.0,NaN,0,20,0.846154,4.0,1,NaN,-4.765625,10.03125,2.0,ltp,NaN,2513.0,3.0,2.0,0,toy_store,-29.187500,-93.875000,Temporal,LTP606,0,15.0
93,2767998,REC_WORD,wordpools/valuecourier_wordpool.txt,14.066667,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,0.0,NAILGUN,138.0,NaN,0,20,0.846154,4.0,1,NaN,-4.765625,10.03125,2.0,ltp,NaN,4005.0,3.0,5.0,0,hardware_store,-8.867188,131.875000,Temporal,LTP606,0,15.0
94,2770338,REC_WORD,wordpools/valuecourier_wordpool.txt,14.066667,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,0.0,JEANS,115.0,NaN,0,20,0.846154,4.0,1,NaN,-4.765625,10.03125,2.0,ltp,NaN,6345.0,3.0,1.0,0,clothing_store,56.718750,77.937500,Temporal,LTP606,0,15.0
95,2772131,REC_WORD,wordpools/valuecourier_wordpool.txt,14.066667,8.461538,/protocols/ltp/subjects/LTP606/experiments/Val...,-1,ValueCourier,NaN,0.0,SYRUP,212.0,NaN,0,20,0.846154,4.0,1,NaN,-4.765625,10.03125,2.0,ltp,NaN,8138.0,3.0,9.0,0,cafe,-112.062500,-30.484375,Temporal,LTP606,0,15.0


**Exercise: What is R1171M's overall recall percent correct for FR1? (Optional)**

**Exercise: What is R1171M's percent correct at each serial position for FR1? Generate a plot with your results (Optional)**